In [ ]:
from pathlib import Path
import subprocess
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/urdu-question-generator')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Areesha-008/Urdu-Question-Generator-.git', str(ROOT)], check=True)
    BASE = Path('/content/drive/MyDrive/Urdu-QG-v2')
else:
    ROOT = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path.cwd().parent
    BASE = ROOT / 'artifacts'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
sys.path.insert(0, str(ROOT))
DATA = BASE / 'data'
RUN = BASE / 'answer_gru_v1'
RUN.mkdir(parents=True, exist_ok=True)
print('Data:', DATA, 'Run:', RUN)


In [ ]:
from scripts.train import train

history = train(DATA, RUN, epochs=2, debug=True)


In [ ]:
history = train(DATA, RUN, epochs=15)


In [ ]:
import matplotlib.pyplot as plt
from app.inference import Generator
from scripts.prepare_data import read_pairs

plt.plot([row['train_loss'] for row in history], label='Train')
plt.plot([row['valid_loss'] for row in history], label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Token cross-entropy'); plt.legend()
plt.savefig(RUN / 'loss_curve.png', dpi=150)
generator = Generator(RUN)
for source, reference in read_pairs(DATA / 'valid.tsv')[::1000][:5]:
    print(source, reference, generator.generate(source)['text'], sep='\n')
